In [1]:
import os
import torch
import librosa
import re
import numpy as np
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor, pipeline

/ihome/lshangguan/zhw227/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# 1. 环境与模型初始化
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"检测到设备: {device} | 精度: {torch_dtype}")

# --- A. 加载分类模型 (AST) ---
ast_model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"
ast_extractor = AutoFeatureExtractor.from_pretrained(ast_model_name)
ast_model = AutoModelForAudioClassification.from_pretrained(ast_model_name).to(device)
ast_model.eval()

# --- B. 加载转录模型 (Distil-Whisper) ---
distil_model_id = "distil-whisper/distil-large-v3"
print(f"正在加载转录模型...")

try:
    transcribe_pipe = pipeline(
        "automatic-speech-recognition",
        model=distil_model_id,
        torch_dtype=torch_dtype,
        device=0 if device == "cuda" else -1,
        model_kwargs={"attn_implementation": "sdpa"} if device == "cuda" else {}
    )
except Exception as e:
    print(f"加速加载失败，尝试基础模式: {e}")
    transcribe_pipe = pipeline(
        "automatic-speech-recognition",
        model=distil_model_id,
        device=0 if device == "cuda" else -1
    )

def get_audio_label(file_path):
    """利用 AST 进行二分类"""
    try:
        y, sr = librosa.load(file_path, sr=16000)
        waveform = torch.from_numpy(y).unsqueeze(0) 
        inputs = ast_extractor(waveform.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = ast_model(**inputs)
        
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        _, top_indices = torch.topk(probs, k=5)
        id2label = ast_model.config.id2label
        top_labels = [id2label[idx.item()] for idx in top_indices[0]]
        
        speech_related = ["Speech", "Conversation", "Babbling", "Child speech"]
        is_speech = any(any(s in label for s in speech_related) for label in top_labels)
        return "Speech" if is_speech else "Environment"
    except Exception as e:
        return f"Error: {e}"

def transcribe_with_distil(file_path):
    """使用 Distil-Whisper 转录"""
    try:
        result = transcribe_pipe(file_path)
        return result["text"].strip()
    except Exception as e:
        return f"[转录失败: {e}]"

# 2. 评测逻辑配置
input_root = "test_experiment_v2"
target_keyword = "bob"
output_txt_file = "transcription_logs.txt" # 结果保存路径
eval_results = []

print(f"\n开始自动化评测目录: {input_root}")

for root, _, files in os.walk(input_root):
    for filename in files:
        if filename.lower().endswith('.wav'):
            file_path = os.path.join(root, filename)
            rel_path = os.path.relpath(file_path, input_root)
            
            should_contain_bob = "true" in root.lower()
            pred_label = get_audio_label(file_path)
            
            transcription = ""
            if pred_label == "Speech":
                transcription = transcribe_with_distil(file_path)
            
            found_bob = re.search(rf'\b{target_keyword}\b', transcription.lower()) is not None
            
            eval_results.append({
                "file": rel_path,
                "gt": should_contain_bob,
                "pred_speech": pred_label == "Speech",
                "pred_keyword": found_bob,
                "text": transcription
            })

# 3. 计算统计指标与保存文件
tp, fp, tn, fn = 0, 0, 0, 0
ast_errors = 0

# 打开文件准备写入
with open(output_txt_file, "w", encoding="utf-8") as f:
    f.write(f"实验评测转录日志 (目标关键词: {target_keyword.upper()})\n")
    f.write("="*80 + "\n")
    f.write(f"{'文件名':<40} | {'真实标签':<10} | {'识别结果':<10} | {'转录内容'}\n")
    f.write("-" * 80 + "\n")

    for res in sorted(eval_results, key=lambda x: x['file']):
        if not res["pred_speech"]:
            ast_errors += 1
        
        # 指标计算逻辑
        if res["gt"] is True:
            if res["pred_keyword"]: tp += 1
            else: fn += 1
        else:
            if res["pred_keyword"]: fp += 1
            else: tn += 1
        
        # 写入 TXT 逻辑
        gt_str = "True" if res["gt"] else "False"
        pred_str = "FOUND" if res["pred_keyword"] else "NOT FOUND"
        if not res["pred_speech"]: pred_str = "NON-SPEECH"
        
        f.write(f"{res['file']:<40} | {gt_str:<10} | {pred_str:<10} | {res['text']}\n")

# 4. 输出汇总报告 (终端显示)
total = len(eval_results)
accuracy = (tp + tn) / total if total > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0

summary = f"""
{"="*60}
实验评测汇总报告
{"="*60}
总样本数: {total} | AST 识别失败: {ast_errors}
准确率 (Accuracy):  {accuracy:.2%}
精确率 (Precision): {precision:.2%}, 召回率 (Recall): {recall:.2%}
------------------------------------------------------------
【TP】: {tp:<3} (成功识别 Bob), 【FN】: {fn:<3} (漏掉 Bob)
【TN】: {tn:<3} (排除他人),     【FP】: {fp:<3} (误报为 Bob)
{"="*60}
转录明细已保存至: {output_txt_file}
"""
print(summary)

检测到设备: cuda | 精度: torch.float16


Loading weights: 100%|██████████| 203/203 [00:00<00:00, 1966.59it/s, Materializing param=classifier.layernorm.weight]                                                    


正在加载转录模型...


Loading weights: 100%|██████████| 539/539 [00:00<00:00, 1979.88it/s, Materializing param=model.encoder.layers.31.self_attn_layer_norm.weight]



开始自动化评测目录: test_experiment_v2

实验评测汇总报告
总样本数: 200 | AST 识别失败: 0
准确率 (Accuracy):  98.00%
精确率 (Precision): 100.00%, 召回率 (Recall): 96.00%
------------------------------------------------------------
【TP】: 96  (成功识别 Bob), 【FN】: 4   (漏掉 Bob)
【TN】: 100 (排除他人),     【FP】: 0   (误报为 Bob)
转录明细已保存至: transcription_logs.txt



In [2]:
# 1. 环境与模型初始化
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"正在启动转录系统... 设备: {device} | 精度: {torch_dtype}")

# --- A. 加载分类模型 (AST) 作为语音过滤器 ---
ast_model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"
ast_extractor = AutoFeatureExtractor.from_pretrained(ast_model_name)
ast_model = AutoModelForAudioClassification.from_pretrained(ast_model_name).to(device)
ast_model.eval()

# --- B. 加载转录模型 (Distil-Whisper) ---
distil_model_id = "distil-whisper/distil-large-v3"
try:
    transcribe_pipe = pipeline(
        "automatic-speech-recognition",
        model=distil_model_id,
        torch_dtype=torch_dtype,
        device=0 if device == "cuda" else -1,
        model_kwargs={"attn_implementation": "sdpa"} if device == "cuda" else {}
    )
except Exception:
    transcribe_pipe = pipeline("automatic-speech-recognition", model=distil_model_id, device=0 if device == "cuda" else -1)

def get_audio_label(file_path):
    """检测是否包含语音"""
    try:
        y, sr = librosa.load(file_path, sr=16000)
        waveform = torch.from_numpy(y).unsqueeze(0) 
        inputs = ast_extractor(waveform.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = ast_model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        _, top_indices = torch.topk(probs, k=5)
        id2label = ast_model.config.id2label
        top_labels = [id2label[idx.item()] for idx in top_indices[0]]
        speech_related = ["Speech", "Conversation", "Babbling", "Child speech"]
        return any(any(s in label for s in speech_related) for label in top_labels)
    except: return False

def transcribe_with_distil(file_path):
    """执行转录"""
    try:
        return transcribe_pipe(file_path)["text"].strip()
    except: return "[转录失败]"

# 2. 核心执行逻辑
input_root = "test_experiment_get_attention" # 输入文件夹
output_txt_file = "raw_transcriptions_get_attention.txt" # 输出文件
final_logs = []

print(f"开始批量转录目录: {input_root} ...")

for root, _, files in os.walk(input_root):
    for filename in files:
        if filename.lower().endswith('.wav'):
            file_path = os.path.join(root, filename)
            rel_path = os.path.relpath(file_path, input_root)
            
            # 过滤与转录
            is_speech = get_audio_label(file_path)
            text = transcribe_with_distil(file_path) if is_speech else "[未检测到语音]"
            
            final_logs.append((rel_path, text))
            if is_speech: print(f"已转录: {rel_path}")

# 3. 保存至文件
with open(output_txt_file, "w", encoding="utf-8") as f:
    f.write(f"{'文件名':<50} | {'转录内容'}\n")
    f.write("-" * 80 + "\n")
    for path, content in sorted(final_logs):
        f.write(f"{path:<50} | {content}\n")

print(f"\n任务完成！所有转录结果已保存至: {output_txt_file}")

正在启动转录系统... 设备: cuda | 精度: torch.float16


Loading weights: 100%|██████████| 203/203 [00:00<00:00, 975.01it/s, Materializing param=classifier.layernorm.weight]                                                   
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 539/539 [00:00<00:00, 864.17it/s, Materializing param=model.encoder.layers.31.self_attn_layer_norm.weight] 


开始批量转录目录: test_experiment_get_attention ...


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


已转录: false/false_text0_voice0.wav
已转录: false/false_text0_voice1.wav
已转录: false/false_text0_voice2.wav
已转录: false/false_text0_voice3.wav
已转录: false/false_text10_voice0.wav
已转录: false/false_text10_voice1.wav
已转录: false/false_text10_voice2.wav
已转录: false/false_text10_voice3.wav
已转录: false/false_text11_voice0.wav


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


已转录: false/false_text11_voice1.wav
已转录: false/false_text11_voice2.wav
已转录: false/false_text11_voice3.wav
已转录: false/false_text12_voice0.wav
已转录: false/false_text12_voice1.wav
已转录: false/false_text12_voice2.wav
已转录: false/false_text12_voice3.wav
已转录: false/false_text13_voice0.wav
已转录: false/false_text13_voice1.wav
已转录: false/false_text13_voice2.wav
已转录: false/false_text13_voice3.wav
已转录: false/false_text14_voice0.wav
已转录: false/false_text14_voice1.wav
已转录: false/false_text14_voice2.wav
已转录: false/false_text14_voice3.wav
已转录: false/false_text15_voice0.wav
已转录: false/false_text15_voice1.wav
已转录: false/false_text15_voice2.wav
已转录: false/false_text15_voice3.wav
已转录: false/false_text16_voice0.wav
已转录: false/false_text16_voice1.wav
已转录: false/false_text16_voice2.wav
已转录: false/false_text16_voice3.wav
已转录: false/false_text17_voice0.wav
已转录: false/false_text17_voice1.wav
已转录: false/false_text17_voice2.wav
已转录: false/false_text17_voice3.wav
已转录: false/false_text18_voice0.wav
已转录: false/false_tex